<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/03_TADP_Experiment_B_Scalability_K20_K50_K100_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


In [ ]:
# ======================================================================================
# 03_TADP_Experiment_B_Scalability_K20_K50_K100_REFINED.py
# TADP Experiment B — Predictive + governance scalability (FINAL admission design)
# ======================================================================================
#
# FINAL GOVERNANCE DESIGN IMPLEMENTED HERE
# ----------------------------------------
# 1) 28 factors across six dimensions (4 + 8 + 4 + 3 + 5 + 4).
# 2) Documentary evidence is verified against predefined 0--5 rubrics.
#    Reviewers verify evidence only; they do NOT assign HPS or make admission decisions.
# 3) Eight Data Quality factors are machine-measured from client TRAIN partitions only.
# 4) Evidence-validity gate: all 28 scores must be present, finite, and > 0.
# 5) Every averaged dimension must be >= 2.5/5.
# 6) HPS < 3.0 -> REJECT.
# 7) HPS >= 3.5 -> direct ACCEPT after the validity/floor gates pass.
# 8) 3.0 <= HPS < 3.5 -> automated critical-dimension HPS_review.
# 9) HPS_review >= 3.25 -> ACCEPT; otherwise REJECT.
# 10) Final decision is binary ACCEPT / REJECT.
#
# REMOVED FROM THE OLD IMPLEMENTATION
# -----------------------------------
# - legacy mandatory-factor gate / factor-specific adequacy minima
# - legacy factor-level Review Score
# - deprecated boundary-selection mode
# - Great Expectations from this scalability notebook (not part of Experiment B)
# - forced 60% admission assertion
#
# EXPERIMENT DESIGN
# -----------------
# Governance overhead: K = 10, 20, 50, 100 (no K=10 retraining).
# Predictive scalability: K = 20, 50, 100, three training seeds, four FL rounds.
# Predictive scenarios: Vanilla FedAvg, matched Random-K, TADP-VR Federated.
# Random-K always matches the OBSERVED TADP admitted-client count and exact optimizer-step
# budget. The code never forces TADP to admit 60%; machine-measured DQ remains authoritative.
#
# TEST IS FINAL-EVALUATION-ONLY.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
import platform
import re
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ======================================================================================
# FINAL POLICY + EXPERIMENT CONSTANTS
# ======================================================================================
GOOD_CUT = 3.0
HIGH_CUT = 3.5
DIMENSION_MIN_FLOOR = 2.5
REVIEW_MIDPOINT = (GOOD_CUT + HIGH_CUT) / 2.0  # 3.25
EVIDENCE_VALIDITY_MIN_EXCLUSIVE = 0.0
MAX_FACTOR_SCORE = 5.0

WEIGHTS_HPS = {
    "dim1": 0.25,
    "dim2": 0.15,
    "dim3": 0.10,
    "dim4": 0.10,
    "dim5": 0.30,
    "dim6": 0.10,
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

FACTOR_NAMES = {
    "dim1": ["source_reputation", "data_controller", "data_objective", "data_collection_lineage"],
    "dim2": [
        "completeness", "duplication_rate", "value_validity_error_rate", "type_consistency",
        "label_integrity", "feature_distribution_consistency", "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": ["data_dictionary", "version_logs", "collection_protocol", "definition_updates"],
    "dim4": ["data_freshness", "scheduled_refresh", "retention_clarity"],
    "dim5": ["regulation_coverage", "consent_ethics", "geo_restrictions", "sensitivity_classification", "audits"],
    "dim6": ["license_terms", "ethical_reviews", "redistribution", "user_agreements"],
}
DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# Four critical dimensions used only to resolve the borderline HPS band.
CRITICAL_REVIEW_DIMS = ("dim1", "dim2", "dim5", "dim6")
_CRITICAL_REVIEW_WEIGHT_TOTAL = sum(WEIGHTS_HPS[d] for d in CRITICAL_REVIEW_DIMS)
CRITICAL_REVIEW_WEIGHTS = {
    d: WEIGHTS_HPS[d] / _CRITICAL_REVIEW_WEIGHT_TOTAL
    for d in CRITICAL_REVIEW_DIMS
}
assert _CRITICAL_REVIEW_WEIGHT_TOTAL > 0.0
assert abs(sum(CRITICAL_REVIEW_WEIGHTS.values()) - 1.0) < 1e-12

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

# Runtime/resource accounting. Energy is estimated from measured runtime.
AVERAGE_COMPUTE_POWER_W = 12.0
ELECTRICITY_PRICE_USD_PER_KWH = 0.20
CARBON_INTENSITY_KG_PER_KWH = 0.445
COMMUNICATION_PRICE_USD_PER_MB = 0.005
MB = 10 ** 6
MIB = 1024 ** 2

# Diabetes 130-US schema constants used by the leakage-safe preparation pipeline.
TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]

EXPERIMENT_VERSION = "TADP-B-FINAL-v18.0-A1A2-OPTIONA-SCALABILITY-K20-50-100-3SEED-4ROUND"
GOVERNANCE_CLIENT_COUNTS = [10, 20, 50, 100]
PREDICTIVE_CLIENT_COUNTS = [20, 50, 100]
TRAINING_RUN_SEEDS = [42, 142, 242]
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DIRICHLET_ALPHA = 1.0
MIN_CLIENT_RECORDS = 30
GLOBAL_SPLIT_SEED = 7001
PARTITION_BASE_SEED = 9101
EVIDENCE_BASE_SEED = 12042
RANDOMK_BASE_SEED = 22042
# K=10 governance audit is anchored to the main Experiment-A configuration.
K10_PARTITION_SEED = 7101
K10_EVIDENCE_SEED = 1042
USE_GOOGLE_DRIVE_CHECKPOINTS = True
SCENARIOS = ["Vanilla FedAvg", "Random-K", "TADP-VR Federated"]

assert abs(sum(WEIGHTS_HPS.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28


RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}


def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])



def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """Mean 0--5 factor score within each of the six HPS dimensions."""
    out = {}
    for dim, expected_factors in FACTOR_NAMES.items():
        vals = []
        valid = True
        for factor in expected_factors:
            if factor not in factors.get(dim, {}):
                valid = False
                break
            try:
                v = float(factors[dim][factor])
            except Exception:
                valid = False
                break
            if not np.isfinite(v):
                valid = False
                break
            vals.append(v)
        out[dim] = float(np.mean(vals)) if valid and vals else np.nan
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_HPS,
) -> float:
    vals = [float(dimensions[d]) for d in weights]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(sum(float(weights[d]) * float(dimensions[d]) for d in weights))


def evidence_validity_audit(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    """
    Gate 1: all 28 expected factor scores must be present, finite, and >0.
    No factor-specific adequacy or mandatory-factor minima are used.
    """
    missing, nonfinite, nonpositive = [], [], []
    scores = {}
    for dim, expected_factors in FACTOR_NAMES.items():
        for factor in expected_factors:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue
            try:
                value = float(factors[dim][factor])
            except Exception:
                nonfinite.append(key)
                continue
            if not np.isfinite(value):
                nonfinite.append(key)
                continue
            scores[key] = value
            if value <= EVIDENCE_VALIDITY_MIN_EXCLUSIVE:
                nonpositive.append(f"{key}:{value:.3f}")
    expected_n = int(sum(len(v) for v in FACTOR_NAMES.values()))
    passed = (
        len(scores) == expected_n
        and not missing
        and not nonfinite
        and not nonpositive
    )
    return {
        "passed": bool(passed),
        "factor_count_expected": expected_n,
        "factor_count_observed_valid_numeric": int(len(scores)),
        "missing": missing,
        "nonfinite": nonfinite,
        "nonpositive": nonpositive,
    }


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def compute_hps_review(dimensions: Dict[str, float]) -> float:
    """
    Borderline-band review over D1,D2,D5,D6 using the ORIGINAL HPS weights
    renormalized over those four critical dimensions.
    """
    vals = [float(dimensions[d]) for d in CRITICAL_REVIEW_DIMS]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(sum(
        CRITICAL_REVIEW_WEIGHTS[d] * float(dimensions[d])
        for d in CRITICAL_REVIEW_DIMS
    ))


def tadp_decision(factors: Dict[str, Dict[str, float]]) -> Dict[str, Any]:
    """
    FINAL A1/A2 TADP admission policy.

    1) All 28 factors present, finite, and >0; otherwise REJECT.
    2) Every dimension >=2.5; otherwise REJECT.
    3) HPS <3.0 -> REJECT; HPS >=3.5 -> direct ACCEPT.
    4) 3.0 <= HPS <3.5 -> HPS_review over D1,D2,D5,D6.
       HPS_review >=3.25 -> ACCEPT; otherwise REJECT.
    """
    validity = evidence_validity_audit(factors)
    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)
    hps_review = compute_hps_review(dims)
    floor_fail = dimension_floor_failures(dims)

    base = {
        "dimensions": dims,
        "hps": float(hps) if np.isfinite(hps) else np.nan,
        "hps_review": float(hps_review) if np.isfinite(hps_review) else np.nan,
        "hps_review_midpoint": float(REVIEW_MIDPOINT),
        "hps_review_margin": (
            float(hps_review - REVIEW_MIDPOINT) if np.isfinite(hps_review) else np.nan
        ),
        "critical_review_dimensions": ";".join(CRITICAL_REVIEW_DIMS),
        "critical_review_weights_json": json.dumps(CRITICAL_REVIEW_WEIGHTS, sort_keys=True),
        "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures": ";".join(floor_fail),
        "evidence_validity_pass": bool(validity["passed"]),
        "evidence_factor_count_expected": int(validity["factor_count_expected"]),
        "evidence_factor_count_observed_valid_numeric": int(
            validity["factor_count_observed_valid_numeric"]
        ),
        "evidence_missing": ";".join(validity["missing"]),
        "evidence_nonfinite": ";".join(validity["nonfinite"]),
        "evidence_nonpositive": ";".join(validity["nonpositive"]),
    }

    if not validity["passed"]:
        failures = validity["missing"] + validity["nonfinite"] + validity["nonpositive"]
        return {
            **base,
            "decision_path": "EVIDENCE_VALIDITY_GATE",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_EVIDENCE_VALIDITY",
            "reason": (
                "At least one of the 28 factor scores is missing, non-finite, or <=0: "
                + ";".join(failures)
            ),
        }

    if floor_fail:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below {DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(f"{d}={dims[d]:.3f}" for d in floor_fail)
            ),
        }

    if hps < GOOD_CUT:
        return {
            **base,
            "decision_path": "LOW_HPS",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
            "reason": f"HPS {hps:.3f} < {GOOD_CUT:.3f}",
        }

    if hps >= HIGH_CUT:
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
            "reason": (
                f"All 28 factors valid; every dimension >= {DIMENSION_MIN_FLOOR:.1f}; "
                f"HPS {hps:.3f} >= {HIGH_CUT:.3f}"
            ),
        }

    if np.isfinite(hps_review) and hps_review >= REVIEW_MIDPOINT:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"HPS {hps:.3f} in [{GOOD_CUT:.3f},{HIGH_CUT:.3f}); "
                f"HPS_review {hps_review:.3f} >= {REVIEW_MIDPOINT:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_HPS_REVIEW",
        "reason": (
            f"HPS {hps:.3f} in [{GOOD_CUT:.3f},{HIGH_CUT:.3f}); "
            f"HPS_review {hps_review:.3f} < {REVIEW_MIDPOINT:.3f}"
        ),
    }


def _documentary_profile(role: str, variant: int = 0) -> Dict[str, Dict[str, float]]:
    """
    Controlled documentary evidence for branch coverage under the FINAL policy.
    DQ (dim2) is never synthesized here; it remains machine-measured from raw TRAIN.
    """
    x = {dim: {} for dim in DOCUMENTARY_DIMS}

    def set_values(dim, values):
        if len(values) != len(FACTOR_NAMES[dim]):
            raise RuntimeError(f"Profile length mismatch for {dim}.")
        for factor, value in zip(FACTOR_NAMES[dim], values):
            x[dim][factor] = float(value)

    if role == "DIRECT_STRONG":
        for dim in DOCUMENTARY_DIMS:
            for j, factor in enumerate(FACTOR_NAMES[dim]):
                x[dim][factor] = float(
                    5.0 if ((j + variant + len(dim)) % 3 == 0) else 4.0
                )

    elif role == "REVIEW_RECOVERABLE":
        # Designed to remain in the global HPS review band while the critical
        # D1/D2/D5/D6 review score clears 3.25 even when DQ is near the 2.5 floor.
        set_values("dim1", [3.0, 4.0, 3.0, 4.0])          # mean 3.5
        set_values("dim3", [2.0, 3.0, 2.0, 3.0])          # mean 2.5
        set_values("dim4", [2.0, 3.0, 2.5])               # mean 2.5
        set_values("dim5", [3.0, 4.0, 3.0, 4.0, 3.5])     # mean 3.5
        set_values("dim6", [3.0, 4.0, 3.0, 4.0])          # mean 3.5
        if variant % 2 == 1:
            x["dim3"]["data_dictionary"] = 3.0

    elif role == "REVIEW_LIMITED":
        set_values("dim1", [2.0, 3.0, 2.0, 3.0])          # mean 2.5
        set_values("dim3", [5.0, 5.0, 5.0, 5.0])          # mean 5.0
        set_values("dim4", [5.0, 5.0, 5.0])               # mean 5.0
        set_values("dim5", [2.0, 3.0, 2.0, 3.0, 3.0])     # mean 2.6
        set_values("dim6", [2.0, 3.0, 2.0, 3.0])          # mean 2.5

    elif role == "EVIDENCE_INVALID":
        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                x[dim][factor] = 4.0
        invalid_candidates = [
            ("dim1", "source_reputation"),
            ("dim3", "data_dictionary"),
            ("dim4", "data_freshness"),
            ("dim5", "geo_restrictions"),
            ("dim6", "ethical_reviews"),
        ]
        weak_dim, weak_factor = invalid_candidates[int(variant) % len(invalid_candidates)]
        x[weak_dim][weak_factor] = 0.0

    elif role == "LOW_HPS_WEAK":
        set_values("dim1", [2.0, 3.0, 2.0, 3.0])
        set_values("dim3", [2.0, 3.0, 2.0, 3.0])
        set_values("dim4", [2.0, 3.0, 3.0])
        set_values("dim5", [2.0, 3.0, 2.0, 3.0, 3.0])
        set_values("dim6", [2.0, 3.0, 2.0, 3.0])

    elif role == "DIMENSION_FLOOR_WEAK":
        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                x[dim][factor] = 3.0
        for factor in FACTOR_NAMES["dim4"]:
            x["dim4"][factor] = 2.0

    else:
        raise ValueError(f"Unknown controlled evidence role: {role}")

    return x


def generate_scalable_controlled_evidence(client_ids: List[str], evidence_seed: int):
    """
    Repeat the same ten-profile documentary mix at K=10,20,50,100 so the
    evidence composition remains comparable as K scales. Final admission is
    NEVER forced; machine-measured DQ remains authoritative.
    """
    if len(client_ids) % 10 != 0:
        raise ValueError("This controlled scalability design uses K values divisible by 10.")

    template = [
        ("DIRECT_STRONG", 0), ("DIRECT_STRONG", 1),
        ("DIRECT_STRONG", 2), ("DIRECT_STRONG", 3),
        ("REVIEW_RECOVERABLE", 0), ("REVIEW_RECOVERABLE", 1),
        ("REVIEW_LIMITED", 0), ("DIMENSION_FLOOR_WEAK", 0),
        ("LOW_HPS_WEAK", 0), ("EVIDENCE_INVALID", 0),
    ]

    evidence_by_client, rows = {}, []
    for block_idx, start in enumerate(range(0, len(client_ids), 10), start=1):
        block = list(client_ids[start:start + 10])
        rng = np.random.default_rng(int(evidence_seed + 1009 * (block_idx - 1)))
        shuffled = [template[i] for i in rng.permutation(len(template))]

        for cid, (role, variant) in zip(block, shuffled):
            factors = _documentary_profile(role, variant)
            evidence_by_client[str(cid)] = {d: dict(v) for d, v in factors.items()}
            bundle_id = f"B{block_idx:02d}-{role}-{variant}"
            for dim in DOCUMENTARY_DIMS:
                for factor, score in factors[dim].items():
                    rows.append({
                        "client": str(cid),
                        "bundle_id": bundle_id,
                        "evidence_profile": role,
                        "scenario_role": role,
                        "profile_variant": int(variant),
                        "scalability_block": int(block_idx),
                        "dimension": dim,
                        "dimension_name": DIMENSION_NAMES[dim],
                        "factor": factor,
                        "verified_rubric_level_0_5": float(score),
                        "server_mapped_score_0_5": float(score),
                        "rubric_descriptor": rubric_descriptor(dim, factor, score),
                        "human_role": "VERIFY_EVIDENCE_ONLY",
                        "admission_decision_by": "SERVER_POLICY",
                        "evidence_validity_rule": "present_finite_and_strictly_gt_0",
                        "factor_valid_for_gate": bool(np.isfinite(score) and float(score) > 0.0),
                        "is_critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                        "evidence_seed": int(evidence_seed),
                        "validation_status": "CONTROLLED_VERIFIED_EVIDENCE",
                    })
    return evidence_by_client, pd.DataFrame(rows)


def validate_expected_profile_mix(evidence_df: pd.DataFrame, k: int) -> Dict[str, int]:
    one = evidence_df[["client", "evidence_profile"]].drop_duplicates("client")
    got = one["evidence_profile"].value_counts().to_dict()
    scale = int(k // 10)
    expected = {
        "DIRECT_STRONG": 4 * scale,
        "REVIEW_RECOVERABLE": 2 * scale,
        "REVIEW_LIMITED": 1 * scale,
        "DIMENSION_FLOOR_WEAK": 1 * scale,
        "LOW_HPS_WEAK": 1 * scale,
        "EVIDENCE_INVALID": 1 * scale,
    }
    if got != expected:
        raise RuntimeError(
            f"Controlled documentary profile mix mismatch at K={k}: "
            f"got={got}, expected={expected}"
        )
    return expected


def build_tadp_governance(client_ids, documentary_evidence, dq_scores, evidence_seed: int) -> pd.DataFrame:
    rows = []
    for cid in client_ids:
        factors = {d: dict(documentary_evidence[cid][d]) for d in DOCUMENTARY_DIMS}
        factors["dim2"] = {f: float(dq_scores[cid][f]) for f in FACTOR_NAMES["dim2"]}
        d = tadp_decision(factors)
        row = {
            "client": str(cid),
            "evidence_seed": int(evidence_seed),
            "hps": float(d["hps"]) if np.isfinite(d["hps"]) else np.nan,
            "hps_review": float(d["hps_review"]) if np.isfinite(d["hps_review"]) else np.nan,
            "hps_review_midpoint": float(REVIEW_MIDPOINT),
            "hps_review_margin": float(d["hps_review_margin"]) if np.isfinite(d["hps_review_margin"]) else np.nan,
            "critical_review_dimensions": d["critical_review_dimensions"],
            "critical_review_weights_json": d["critical_review_weights_json"],
            "evidence_validity_pass": bool(d["evidence_validity_pass"]),
            "evidence_missing": d["evidence_missing"],
            "evidence_nonfinite": d["evidence_nonfinite"],
            "evidence_nonpositive": d["evidence_nonpositive"],
            "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
            "dimension_floor_failures": d["dimension_floor_failures"],
            "decision_path": d["decision_path"],
            "status": d["status"],
            "final_action": d["final_action"],
            "reason": d["reason"],
        }
        for dim, value in d["dimensions"].items():
            row[f"{dim}_score_0_5"] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def build_full_factor_evidence_table(client_ids, documentary_evidence, dq_scores, dq_audit, evidence_seed):
    dq_idx = dq_audit.copy().set_index("client", drop=False)
    rows = []
    for cid in client_ids:
        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(documentary_evidence[cid][dim][factor])
                rows.append({
                    "client": str(cid),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value": np.nan,
                    "rubric_score_0_5": score,
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "evidence_validity_rule": "present_finite_and_strictly_gt_0",
                    "factor_valid_for_gate": bool(np.isfinite(score) and score > 0.0),
                    "is_critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                    "evidence_seed": int(evidence_seed),
                })
        for factor in FACTOR_NAMES["dim2"]:
            score = float(dq_scores[cid][factor])
            raw_col = DQ_RAW_METRIC_BY_FACTOR[factor]
            raw_value = float(dq_idx.loc[cid, raw_col]) if raw_col in dq_idx.columns else np.nan
            rows.append({
                "client": str(cid),
                "dimension": "dim2",
                "dimension_name": DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source": "MACHINE_MEASURED_RAW_TRAIN",
                "raw_measured_value": raw_value,
                "rubric_score_0_5": score,
                "rubric_descriptor": rubric_descriptor("dim2", factor, score),
                "evidence_validity_rule": "present_finite_and_strictly_gt_0",
                "factor_valid_for_gate": bool(np.isfinite(score) and score > 0.0),
                "is_critical_review_dimension": True,
                "evidence_seed": int(evidence_seed),
            })
    out = pd.DataFrame(rows)
    expected = len(client_ids) * 28
    if len(out) != expected:
        raise RuntimeError(f"Expected {expected} factor-evidence rows, found {len(out)}")
    return out


def policy_self_test():
    # Use deterministic synthetic DQ=4 to verify the final policy branches.
    roles = [
        "DIRECT_STRONG", "REVIEW_RECOVERABLE", "REVIEW_LIMITED",
        "DIMENSION_FLOOR_WEAK", "LOW_HPS_WEAK", "EVIDENCE_INVALID",
    ]
    observed = {}
    for role in roles:
        factors = _documentary_profile(role, 0)
        factors["dim2"] = {f: 4.0 for f in FACTOR_NAMES["dim2"]}
        d = tadp_decision(factors)
        observed[role] = (d["decision_path"], d["final_action"])

    expected = {
        "DIRECT_STRONG": ("DIRECT_AUTO_ACCEPT", "ACCEPT"),
        "REVIEW_RECOVERABLE": ("HPS_REVIEW_BAND", "ACCEPT"),
        "REVIEW_LIMITED": ("HPS_REVIEW_BAND", "REJECT"),
        "DIMENSION_FLOOR_WEAK": ("DIMENSION_FLOOR", "REJECT"),
        "LOW_HPS_WEAK": ("LOW_HPS", "REJECT"),
        "EVIDENCE_INVALID": ("EVIDENCE_VALIDITY_GATE", "REJECT"),
    }
    if observed != expected:
        raise RuntimeError(f"Final policy self-test failed: observed={observed}, expected={expected}")

    # Explicitly verify the published critical review weights and threshold.
    expected_weights = {
        "dim1": 0.3125, "dim2": 0.1875, "dim5": 0.375, "dim6": 0.125
    }
    for d, v in expected_weights.items():
        if not np.isclose(CRITICAL_REVIEW_WEIGHTS[d], v):
            raise RuntimeError(
                f"Critical review weight mismatch for {d}: "
                f"{CRITICAL_REVIEW_WEIGHTS[d]} != {v}"
            )
    if not np.isclose(REVIEW_MIDPOINT, 3.25):
        raise RuntimeError("HPS_review threshold must be 3.25.")
    return True


POLICY_SELF_TEST_PASS = policy_self_test()




def write_tadp_append_only_ledger(gov: pd.DataFrame, output_path: Path, evidence_version: str):
    """
    Prototype append-only CSV with a SHA-256 forward hash chain.
    Ledger write time covers CSV serialization, flush, and an fsync request.
    This is tamper-evident prototype storage, not hardened immutable storage.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()
    trace_path = output_path.with_name(output_path.stem + "_growth_trace.csv")
    if trace_path.exists():
        trace_path.unlink()

    prev_hash = "GENESIS"
    trace_rows = []
    fieldnames = None
    start = time.perf_counter()

    import csv
    with open(output_path, "w", newline="", encoding="utf-8") as fh:
        writer = None
        for seq, (_, r) in enumerate(gov.sort_values("client").iterrows(), start=1):
            dims = {d: float(r[f"{d}_score_0_5"]) for d in FACTOR_NAMES}
            payload = {
                "sequence": int(seq),
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "client": str(r["client"]),
                "evidence_version": str(evidence_version),
                "dimension_scores_json": json.dumps(dims, sort_keys=True),
                "hps": float(r["hps"]) if np.isfinite(r["hps"]) else np.nan,
                "hps_review": float(r["hps_review"]) if np.isfinite(r["hps_review"]) else np.nan,
                "evidence_validity_pass": bool(r["evidence_validity_pass"]),
                "dimension_floor_failures": str(r["dimension_floor_failures"]),
                "decision_path": str(r["decision_path"]),
                "status": str(r["status"]),
                "final_action": str(r["final_action"]),
                "previous_hash": prev_hash,
            }
            canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
            entry_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
            payload["entry_hash"] = entry_hash

            if writer is None:
                fieldnames = list(payload.keys())
                writer = csv.DictWriter(fh, fieldnames=fieldnames)
                writer.writeheader()
            writer.writerow(payload)
            fh.flush()
            os.fsync(fh.fileno())

            prev_hash = entry_hash
            trace_rows.append({
                "sequence": int(seq),
                "client": str(r["client"]),
                "ledger_bytes_after_append": int(output_path.stat().st_size),
                "entry_hash": entry_hash,
            })

    elapsed = float(time.perf_counter() - start)
    pd.DataFrame(trace_rows).to_csv(trace_path, index=False)
    return {
        "ledger_entries": int(len(gov)),
        "ledger_bytes": int(output_path.stat().st_size),
        "ledger_write_s": elapsed,
        "ledger_write_semantics": "CSV serialization + flush + fsync request",
        "ledger_path": str(output_path),
        "ledger_growth_trace_path": str(trace_path),
    }

def print_banner(title: str, width: int = 108):
    """Compact console section banner used by the Colab experiment driver."""
    print("\n" + "=" * int(width))
    print(str(title))
    print("=" * int(width))


def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()

def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()

def _process_rss_mib() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0

class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mib = 0.0
        self.end_mib = 0.0
        self.peak_mib = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mib = max(self.peak_mib, _process_rss_mib())

    def start(self):
        self.start_mib = _process_rss_mib()
        self.peak_mib = self.start_mib
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mib = _process_rss_mib()
        self.peak_mib = max(self.peak_mib, self.start_mib, self.end_mib)
        return {
            "ram_start_mib": float(self.start_mib),
            "ram_end_mib": float(self.end_mib),
            "ram_peak_mib": float(self.peak_mib),
            "ram_delta_mib": float(max(0.0, self.peak_mib - self.start_mib)),
            "ram_mib": float(self.peak_mib),
        }

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)

def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )

def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MiB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path

def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0

def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0

def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}

def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))

def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out

def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = 4096,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]

def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"

def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mib: float,
    ram_start_mib: float,
    ram_peak_mib: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        AVERAGE_COMPUTE_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mib) - float(ram_start_mib))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy≈{energy_wh:.5f}Wh | communication={cumulative_communication_mib:.3f} MiB | "
        f"RAM peak={ram_peak_mib:.1f} MiB | RAM Δ={ram_delta:.1f} MiB"
    )
    print("-" * 112)

def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))

def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))

def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc

def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out

def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Logical application-level model payload: one model download and
        # one model-update upload per participating client. No universal
        # transport/protocol-overhead multiplier is assumed.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mib = float(cumulative_comm_raw_b / MIB)

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mib)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                AVERAGE_COMPUTE_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mib": float(cumulative_comm_mib),
            "ram_peak_mib_live": float(live_peak),
            "ram_delta_mib_live": float(max(0.0, live_peak - monitor.start_mib)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mib=cumulative_comm_mib,
            ram_start_mib=float(monitor.start_mib),
            ram_peak_mib=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_bytes": int(communication_b),
        "communication_mb": float(communication_b / MB),
        "communication_mib": float(communication_b / MIB),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }


def result_row(run: int, seed: int, scenario: str, result: Dict[str, Any], initial_hash: str) -> Dict[str, Any]:
    row = {
        "run": int(run), "seed": int(seed), "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_bytes": int(result.get("communication_bytes", 0)),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mib": float(result.get("ram_start_mib", 0.0)),
        "ram_end_mib": float(result.get("ram_end_mib", 0.0)),
        "ram_peak_mib": float(result.get("ram_peak_mib", 0.0)),
        "ram_delta_mib": float(result.get("ram_delta_mib", 0.0)),
        "initial_weights_sha256": str(initial_hash),
    }
    row["estimated_energy_wh"] = AVERAGE_COMPUTE_POWER_W * row["runtime_s"] / 3600.0
    row["estimated_energy_kwh"] = row["estimated_energy_wh"] / 1000.0
    row["estimated_co2_kg"] = row["estimated_energy_kwh"] * CARBON_INTENSITY_KG_PER_KWH
    row["estimated_energy_cost_usd"] = row["estimated_energy_kwh"] * ELECTRICITY_PRICE_USD_PER_KWH
    row["normalized_energy_wh_per_w"] = row["runtime_s"] / 3600.0
    row["communication_cost_usd"] = row["communication_mb"] * COMMUNICATION_PRICE_USD_PER_MB
    row["estimated_total_cost_usd"] = row["estimated_energy_cost_usd"] + row["communication_cost_usd"]
    return row


def locate_diabetes_csv() -> str:
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "/content/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]
    for p in candidates:
        if p and os.path.exists(p):
            return p

    try:
        from google.colab import files as colab_files
        print("Please upload the Diabetes 130-US CSV file.")
        uploaded = colab_files.upload()
        csvs = [name for name in uploaded if str(name).lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError("No CSV file was uploaded.")
        return str(csvs[0])
    except ImportError:
        pass

    raise FileNotFoundError(
        "Diabetes CSV not found. Set DIABETES_CSV or upload the CSV in Colab."
    )

def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df

def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


@dataclass
class TabularDQSchema:
    """
    Governance-only raw-TRAIN schema used for Data Quality measurement before
    admission. It contains no model scaling parameters, model vocabulary,
    class weights, or transformed arrays.
    """
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]


def infer_train_only_dq_schema(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularDQSchema:
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]
    numeric_cols, categorical_cols = [], []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        (numeric_cols if ratio >= 0.95 else categorical_cols).append(c)
    return TabularDQSchema(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
    )


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


# Fail early if source/notebook assembly ever corrupts the dataclass definition.
if not hasattr(TabularPreprocessor, "__dataclass_fields__"):
    raise RuntimeError("TabularPreprocessor must be decorated with @dataclass.")
if not callable(getattr(TabularPreprocessor, "transform", None)):
    raise RuntimeError("TabularPreprocessor.transform() is missing from the class definition.")

def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def preprocessor_fingerprint(pre: TabularPreprocessor) -> str:
    payload = {
        "feature_cols": list(pre.feature_cols),
        "numeric_cols": list(pre.numeric_cols),
        "categorical_cols": list(pre.categorical_cols),
        "mean": {k: float(v) for k, v in pre.mean.items()},
        "std": {k: float(v) for k, v in pre.std.items()},
        "categories": {k: list(map(str, v)) for k, v in pre.categories.items()},
    }
    canonical = json.dumps(
        payload, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest()


def build_post_selection_training_view(
    clients_raw: Dict[str, pd.DataFrame],
    test_df: pd.DataFrame,
    participating_clients: List[str],
    view_name: str,
) -> Dict[str, Any]:
    """
    OPTION A: fit model preprocessing and class weights only AFTER the scenario's
    participating cohort has been frozen.
    """
    participating_clients = list(map(str, participating_clients))
    if not participating_clients:
        raise RuntimeError(f"Training view {view_name!r} has no participating clients.")

    selected_frames = {cid: clients_raw[cid] for cid in participating_clients}
    pre = fit_federated_train_only_preprocessor(selected_frames)

    client_arrays = {}
    for cid in participating_clients:
        df = clients_raw[cid]
        client_arrays[cid] = (
            pre.transform(df),
            df["_target"].to_numpy(dtype=np.int32),
        )

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)
    y_train = np.concatenate([client_arrays[cid][1] for cid in participating_clients])
    cw = class_weight_dict(y_train)

    return {
        "view_name": str(view_name),
        "participating_clients": participating_clients,
        "participating_client_count": int(len(participating_clients)),
        "training_rows": int(sum(len(clients_raw[cid]) for cid in participating_clients)),
        "preprocessor": pre,
        "preprocessor_sha256": preprocessor_fingerprint(pre),
        "client_arrays": client_arrays,
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": cw,
        "input_dim": int(X_test.shape[1]),
        "numeric_feature_count": int(len(pre.numeric_cols)),
        "categorical_feature_count": int(len(pre.categorical_cols)),
        "categorical_encoded_width": int(X_test.shape[1] - len(pre.numeric_cols)),
    }


def build_training_view_cache(
    clients_raw: Dict[str, pd.DataFrame],
    test_df: pd.DataFrame,
    cohorts: Dict[str, List[str]],
) -> Tuple[Dict[str, Dict[str, Any]], pd.DataFrame]:
    cache, views, audit_rows = {}, {}, []
    for scenario, cohort in cohorts.items():
        key = tuple(sorted(map(str, cohort)))
        if key not in cache:
            cache[key] = build_post_selection_training_view(
                clients_raw=clients_raw,
                test_df=test_df,
                participating_clients=list(key),
                view_name=";".join(key),
            )
        view = cache[key]
        views[scenario] = view
        audit_rows.append({
            "scenario": str(scenario),
            "participating_clients": ";".join(view["participating_clients"]),
            "participating_client_count": int(view["participating_client_count"]),
            "training_rows": int(view["training_rows"]),
            "input_dim": int(view["input_dim"]),
            "numeric_feature_count": int(view["numeric_feature_count"]),
            "categorical_feature_count": int(view["categorical_feature_count"]),
            "categorical_encoded_width": int(view["categorical_encoded_width"]),
            "preprocessor_sha256": str(view["preprocessor_sha256"]),
            "class_weights_json": json.dumps(
                {str(k): float(v) for k, v in view["class_weights"].items()},
                sort_keys=True,
            ),
            "preprocessing_fit_after_selection": True,
            "unselected_clients_in_model_preprocessing": False,
        })
    return views, pd.DataFrame(audit_rows)


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: TabularPreprocessor,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: TabularPreprocessor,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root

def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)

def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))

def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)

def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)

def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half

def scalable_client_ids(n_clients: int) -> List[str]:
    return [f"C{i:03d}" for i in range(1, int(n_clients) + 1)]

def dirichlet_partition_dataframe_scalable(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 30,
    max_attempts: int = 500,
) -> Dict[str, pd.DataFrame]:
    """
    Label-wise Dirichlet partition that supports K > 26 while preserving every
    global TRAIN row exactly once.
    """
    y = train_df["_target"].to_numpy()
    client_ids = scalable_client_ids(n_clients)

    for attempt in range(int(max_attempts)):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(int(n_clients))]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(int(n_clients), float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for j, split in enumerate(splits):
                buckets[j].extend(split.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            return {
                cid: train_df.iloc[np.asarray(idxs, dtype=int)].copy()
                for cid, idxs in zip(client_ids, buckets)
            }

    raise RuntimeError(
        f"Could not obtain valid K={n_clients} partition after {max_attempts} attempts "
        f"with min_client_records={min_client_records}."
    )

def partition_integrity_audit(
    train_df: pd.DataFrame,
    clients: Dict[str, pd.DataFrame],
) -> Dict[str, Any]:
    global_ids = set(train_df["_row_id"].astype(int).tolist())
    all_ids = []
    for df in clients.values():
        all_ids.extend(df["_row_id"].astype(int).tolist())
    client_set = set(all_ids)

    result = {
        "global_train_rows": int(len(train_df)),
        "client_rows_total": int(len(all_ids)),
        "unique_client_rows": int(len(client_set)),
        "rows_missing_from_clients": int(len(global_ids - client_set)),
        "rows_not_in_global_train": int(len(client_set - global_ids)),
        "duplicate_rows_across_clients": int(len(all_ids) - len(client_set)),
    }
    result["pass"] = bool(
        len(all_ids) == len(train_df)
        and client_set == global_ids
        and len(all_ids) == len(client_set)
    )
    return result




def prepare_k_governance_stage(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    k: int,
    partition_seed: int,
):
    """
    FINAL OPTION-A ordering for Experiment B:
      1) GLOBAL TEST is already isolated.
      2) Partition only raw GLOBAL TRAIN into K client shards.
      3) Do not fit model preprocessing or class weights here.
      4) Governance/DQ is executed next on raw TRAIN shards.
    """
    clients = dirichlet_partition_dataframe_scalable(
        train_df=train_df,
        n_clients=k,
        alpha=DIRICHLET_ALPHA,
        seed=partition_seed,
        min_client_records=MIN_CLIENT_RECORDS,
    )
    integrity = partition_integrity_audit(train_df, clients)
    if not integrity["pass"]:
        raise RuntimeError(f"Partition integrity failed at K={k}: {integrity}")

    return {
        "clients_raw": clients,
        "client_ids": list(clients.keys()),
        "test_df": test_df,
        "y_test": test_df["_target"].to_numpy(dtype=np.int32),
        "integrity": integrity,
        "model_preprocessing_fitted_before_selection": False,
        "class_weights_computed_before_selection": False,
    }


def run_governance_for_k(data: Dict[str, Any], k: int, evidence_seed: int, output_dir: Path):
    monitor = RAMMonitor().start()

    # DQ is measured from raw TRAIN shards before admission and before any
    # model preprocessing is fitted.
    t0 = time.perf_counter()
    dq_schema = infer_train_only_dq_schema(data["clients_raw"])
    dq_reference = build_tabular_reference(data["clients_raw"], dq_schema)
    dq_scores, dq_rows = {}, []
    for cid, df in data["clients_raw"].items():
        scores, raw = tabular_dq_scores(df, dq_schema, dq_reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": str(cid), **scores, **raw})
    dq_processing_s = float(time.perf_counter() - t0)
    dq_audit = pd.DataFrame(dq_rows)

    t1 = time.perf_counter()
    documentary, evidence_df = generate_scalable_controlled_evidence(
        data["client_ids"], evidence_seed,
    )
    expected_profile_mix = validate_expected_profile_mix(evidence_df, k)
    evidence_processing_s = float(time.perf_counter() - t1)

    t2 = time.perf_counter()
    gov = build_tadp_governance(
        data["client_ids"], documentary, dq_scores, evidence_seed=evidence_seed,
    ).sort_values("client").reset_index(drop=True)
    policy_decision_s = float(time.perf_counter() - t2)

    ledger_info = write_tadp_append_only_ledger(
        gov,
        Path(output_dir) / f"tadp_governance_ledger_K{k:03d}.csv",
        evidence_version=f"controlled-final-policy-K{k}-seed{evidence_seed}",
    )
    ram = monitor.stop()

    full_evidence = build_full_factor_evidence_table(
        data["client_ids"], documentary, dq_scores, dq_audit, evidence_seed,
    )

    observed = gov["decision_path"].value_counts().to_dict()
    admitted = accepted_tadp_vr(gov)
    total_governance_s = (
        dq_processing_s + evidence_processing_s + policy_decision_s + ledger_info["ledger_write_s"]
    )
    metrics = {
        "k_submissions": int(k),
        "factor_count": 28,
        "critical_review_dimension_count": int(len(CRITICAL_REVIEW_DIMS)),
        "dq_processing_s": dq_processing_s,
        "controlled_evidence_processing_s": evidence_processing_s,
        "policy_decision_s": policy_decision_s,
        "ledger_write_s": float(ledger_info["ledger_write_s"]),
        "governance_processing_s": float(total_governance_s),
        "ledger_entries": int(ledger_info["ledger_entries"]),
        "ledger_bytes": int(ledger_info["ledger_bytes"]),
        "ledger_bytes_per_entry": float(ledger_info["ledger_bytes"] / max(1, ledger_info["ledger_entries"])),
        "governance_ram_peak_mib": float(ram["ram_peak_mib"]),
        "governance_ram_delta_mib": float(ram["ram_delta_mib"]),
        "admitted_count": int(len(admitted)),
        "admission_rate": float(len(admitted) / max(1, k)),
        "expected_documentary_profile_mix": json.dumps(expected_profile_mix, sort_keys=True),
        "observed_decision_path_counts": json.dumps(observed, sort_keys=True),
        "human_verification_time_included": False,
    }
    return {
        "dq_scores": dq_scores, "dq_audit": dq_audit,
        "documentary": documentary, "evidence_df": evidence_df,
        "governance_df": gov, "admitted_clients": admitted,
        "full_evidence": full_evidence, "metrics": metrics,
    }



def summarize_b3(perf: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr_macro",
        "runtime_s", "communication_bytes", "communication_mb", "communication_mib", "optimizer_steps", "participants",
        "normalized_energy_wh_per_w", "estimated_energy_wh", "estimated_co2_kg", "estimated_energy_cost_usd",
        "communication_cost_usd", "estimated_total_cost_usd", "ram_peak_mib", "ram_delta_mib",
    ]
    rows = []
    for (k, scenario), d in perf.groupby(["k_submissions", "scenario"], sort=True):
        row = {"k_submissions": int(k), "scenario": str(scenario), "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(np.mean(vals)) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_b3(perf: pd.DataFrame) -> pd.DataFrame:
    rows = []
    comparisons = [
        ("Random-K", "MATCHED_SELECTION_CONTROL"),
        ("Vanilla FedAvg", "FULL_PARTICIPATION_REFERENCE"),
    ]
    for k in sorted(perf["k_submissions"].unique()):
        tadp = perf[(perf["k_submissions"].eq(k)) & perf["scenario"].eq("TADP-VR Federated")].copy()
        for baseline, role in comparisons:
            base = perf[(perf["k_submissions"].eq(k)) & perf["scenario"].eq(baseline)].copy()
            merged = tadp.merge(base, on=["k_submissions", "seed"], suffixes=("_tadp", "_baseline"), validate="one_to_one")
            for metric in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr_macro"]:
                a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
                b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
                diff = a - b
                lo, hi = ci95_mean(diff)
                rows.append({
                    "k_submissions": int(k), "baseline": baseline, "baseline_role": role,
                    "metric": metric, "n_pairs": int(len(diff)),
                    "mean_difference_tadp_minus_baseline": float(np.mean(diff)),
                    "sd_difference": float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0,
                    "ci95_low": lo, "ci95_high": hi,
                    "tadp_wins": int(np.sum(diff > 0)), "ties": int(np.sum(np.isclose(diff, 0.0))),
                    "tadp_losses": int(np.sum(diff < 0)),
                })
    return pd.DataFrame(rows)


def utility_retention_table(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in sorted(summary["k_submissions"].unique()):
        d = summary[summary["k_submissions"].eq(k)].set_index("scenario")
        required = {"TADP-VR Federated", "Vanilla FedAvg", "Random-K"}
        if not required.issubset(d.index):
            continue
        row = {"k_submissions": int(k)}
        for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
            tadp = float(d.loc["TADP-VR Federated", f"{metric}_mean"])
            full = float(d.loc["Vanilla FedAvg", f"{metric}_mean"])
            rand = float(d.loc["Random-K", f"{metric}_mean"])
            row[f"tadp_{metric}_mean"] = tadp
            row[f"fedavg_{metric}_mean"] = full
            row[f"randomk_{metric}_mean"] = rand
            row[f"tadp_minus_fedavg_{metric}"] = tadp - full
            row[f"tadp_minus_randomk_{metric}"] = tadp - rand
        row["tadp_participants_mean"] = float(d.loc["TADP-VR Federated", "participants_mean"])
        row["fedavg_participants_mean"] = float(d.loc["Vanilla FedAvg", "participants_mean"])
        row["randomk_participants_mean"] = float(d.loc["Random-K", "participants_mean"])
        tadp_comm = float(d.loc["TADP-VR Federated", "communication_mib_mean"])
        fedavg_comm = float(d.loc["Vanilla FedAvg", "communication_mib_mean"])
        row["tadp_communication_mib_mean"] = tadp_comm
        row["fedavg_communication_mib_mean"] = fedavg_comm
        row["communication_reduction_pct_vs_fedavg"] = 100.0 * (fedavg_comm - tadp_comm) / max(fedavg_comm, 1e-12)
        rows.append(row)
    return pd.DataFrame(rows)


def write_b3_scenario_checkpoint(root: Path, k: int, run_idx: int, scenario: str, result: Dict[str, Any]):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"K_{int(k):03d}" / f"run_{int(run_idx):02d}")
    payload = {
        "k_submissions": int(k), "run": int(run_idx), "scenario": str(scenario),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"], "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "ram_peak_mib": float(result.get("ram_peak_mib", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(cdir / f"{safe}_rounds.csv", index=False)



def partition_seed_for_k(k: int) -> int:
    return K10_PARTITION_SEED if int(k) == 10 else int(PARTITION_BASE_SEED + 101 * int(k))


def evidence_seed_for_k(k: int) -> int:
    return K10_EVIDENCE_SEED if int(k) == 10 else int(EVIDENCE_BASE_SEED + 103 * int(k))


def randomk_seed_for_k(k: int) -> int:
    return int(RANDOMK_BASE_SEED + 107 * int(k))



def save_environment_metadata(root: Path):
    meta = {
        "experiment_version": EXPERIMENT_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "governance_client_counts": GOVERNANCE_CLIENT_COUNTS,
        "predictive_client_counts": PREDICTIVE_CLIENT_COUNTS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "final_policy": {
            "evidence_validity_rule": "all_28_present_finite_and_strictly_gt_0",
            "dimension_floor": DIMENSION_MIN_FLOOR,
            "hps_reject_below": GOOD_CUT,
            "hps_direct_accept_at_or_above": HIGH_CUT,
            "hps_review_threshold": REVIEW_MIDPOINT,
            "critical_review_dimensions": list(CRITICAL_REVIEW_DIMS),
            "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
        },
        "option_a_preprocessing": {
            "governance_before_model_preprocessing": True,
            "scenario_specific_preprocessing_after_participant_selection": True,
            "scenario_specific_class_weights_after_participant_selection": True,
            "test_used_for_preprocessing": False,
        },
        "resource_assumptions": {
            "average_compute_power_w_reference": AVERAGE_COMPUTE_POWER_W,
            "electricity_price_usd_per_kwh_reference": ELECTRICITY_PRICE_USD_PER_KWH,
            "carbon_intensity_kg_per_kwh_reference": CARBON_INTENSITY_KG_PER_KWH,
            "communication_price_usd_per_mb_reference": COMMUNICATION_PRICE_USD_PER_MB,
            "communication_accounting": "logical_application_payload_actual_model_array_bytes",
            "protocol_overhead_multiplier": None,
        },
        "human_verification_time_in_governance_runtime": False,
        "test_semantics": (
            "Held-out TEST is isolated before client partitioning and used only "
            "after training for final predictive evaluation."
        ),
    }
    atomic_write_json(meta, root / "experiment_design.json")


def startup_symbol_self_check():
    required = [
        "TARGET", "ID_COLUMNS", "print_banner", "locate_diabetes_csv",
        "load_diabetes", "global_patient_grouped_split",
        "prepare_k_governance_stage", "run_governance_for_k",
        "build_training_view_cache", "federated_train",
        "package_and_download_results",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Notebook/source assembly is incomplete; missing symbols: "
            + ", ".join(missing)
        )
    return True


def main():
    startup_symbol_self_check()
    print_banner(EXPERIMENT_VERSION)
    print(f"Governance K values: {GOVERNANCE_CLIENT_COUNTS}")
    print(f"Predictive K values: {PREDICTIVE_CLIENT_COUNTS}")
    print(f"Training seeds: {TRAINING_RUN_SEEDS}")
    print(f"FL rounds: {NUM_ROUNDS_FL}")
    print(
        "Final policy: evidence-validity gate -> dimension floor -> HPS bands "
        "-> critical-dimension HPS_review -> binary ACCEPT/REJECT"
    )
    print(
        "Option A: governance/selection precedes model preprocessing; "
        "each scenario fits preprocessing and class weights only from its "
        "participating TRAIN cohort."
    )
    print("K=10 is governance-only here; predictive K=10 remains Experiment A1.")

    root = choose_experiment_root(
        "TADP_EXPERIMENT_B_" + EXPERIMENT_VERSION,
        use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
    )
    checkpoint_state = root / "checkpoint_state.json"
    perf_checkpoint = root / "performance_metrics_checkpoint.csv"
    save_environment_metadata(root)

    csv_path = locate_diabetes_csv()
    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, GLOBAL_SPLIT_SEED
    )
    split_audit = {
        "raw_rows": int(len(raw)),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "patient_overlap": int(patient_overlap),
        "global_split_seed": int(GLOBAL_SPLIT_SEED),
        "pass": bool(int(patient_overlap) == 0),
    }
    atomic_write_json(split_audit, root / "global_split_audit.json")
    if not split_audit["pass"]:
        raise RuntimeError(f"Global split leakage audit failed: {split_audit}")

    completed = set(
        load_checkpoint_state(checkpoint_state).get("completed", [])
    )
    governance_metric_rows = []
    k_design_rows = []
    parity_rows = []
    all_preprocessing_audits = []

    for k in GOVERNANCE_CLIENT_COUNTS:
        print_banner(f"GOVERNANCE / SCALABILITY PREPARATION — K={k}")
        partition_seed = partition_seed_for_k(k)
        evidence_seed = evidence_seed_for_k(k)

        # Raw TRAIN partition + governance-stage DQ only.
        data = prepare_k_governance_stage(
            train_df=train_df,
            test_df=test_df,
            k=k,
            partition_seed=partition_seed,
        )

        part_rows = []
        for cid in data["client_ids"]:
            y = data["clients_raw"][cid]["_target"].to_numpy(dtype=np.int32)
            part_rows.append({
                "client": cid,
                "records": int(len(y)),
                "class_0": int(np.sum(y == 0)),
                "class_1": int(np.sum(y == 1)),
                "class_2": int(np.sum(y == 2)),
            })
        partition_df = pd.DataFrame(part_rows)
        partition_df.to_csv(
            root / f"client_partition_K{k:03d}.csv", index=False
        )

        govrun = run_governance_for_k(
            data, k, evidence_seed, root
        )
        govrun["dq_audit"].to_csv(
            root / f"dq_audit_K{k:03d}.csv", index=False
        )
        govrun["evidence_df"].to_csv(
            root / f"controlled_evidence_K{k:03d}.csv", index=False
        )
        govrun["governance_df"].to_csv(
            root / f"tadp_governance_K{k:03d}.csv", index=False
        )
        govrun["full_evidence"].to_csv(
            root / f"all_28_factor_evidence_K{k:03d}.csv", index=False
        )

        governance_metric_rows.append({
            **govrun["metrics"],
            "partition_seed": int(partition_seed),
            "evidence_seed": int(evidence_seed),
            "min_client_records": int(partition_df["records"].min()),
            "max_client_records": int(partition_df["records"].max()),
            "mean_client_records": float(partition_df["records"].mean()),
            "partition_integrity_pass": bool(data["integrity"]["pass"]),
            "model_preprocessing_fitted_before_selection": False,
            "class_weights_computed_before_selection": False,
        })

        vr_clients = list(govrun["admitted_clients"])
        print(
            f"K={k}: observed TADP admission={len(vr_clients)}/{k} "
            f"({len(vr_clients)/k:.1%}); "
            f"governance={govrun['metrics']['governance_processing_s']:.4f}s; "
            f"ledger={govrun['metrics']['ledger_bytes']} bytes."
        )

        # K=10 contributes governance/ledger scalability only.
        if k not in PREDICTIVE_CLIENT_COUNTS:
            continue
        if not vr_clients:
            raise RuntimeError(
                f"TADP admitted zero clients at K={k}; "
                "predictive training cannot proceed."
            )

        client_ids = list(data["client_ids"])
        all_clients = list(client_ids)

        # Matched Random-K uses the OBSERVED TADP admitted count.
        randomk_seed = randomk_seed_for_k(k)
        rng = np.random.default_rng(randomk_seed)
        randomk_clients = rng.choice(
            client_ids, size=len(vr_clients), replace=False
        ).tolist()

        scenario_clients = {
            "Vanilla FedAvg": all_clients,
            "Random-K": randomk_clients,
            "TADP-VR Federated": vr_clients,
        }

        # FINAL Option A: fit model preprocessing/class weights only now,
        # after each scenario's participating cohort is known.
        scenario_views, preprocessing_audit = build_training_view_cache(
            clients_raw=data["clients_raw"],
            test_df=data["test_df"],
            cohorts=scenario_clients,
        )
        preprocessing_audit.insert(0, "k_submissions", int(k))
        preprocessing_audit["partition_seed"] = int(partition_seed)
        preprocessing_audit["evidence_seed"] = int(evidence_seed)
        preprocessing_audit.to_csv(
            root / f"scenario_preprocessing_audit_K{k:03d}.csv",
            index=False,
        )
        all_preprocessing_audits.append(preprocessing_audit)

        def natural_map(scenario):
            view = scenario_views[scenario]
            return {
                cid: natural_steps(
                    len(view["client_arrays"][cid][1]),
                    BATCH_SIZE,
                    LOCAL_EPOCHS,
                )
                for cid in scenario_clients[scenario]
            }

        step_map_all = natural_map("Vanilla FedAvg")
        step_map_vr = natural_map("TADP-VR Federated")
        vr_target_per_round = int(sum(step_map_vr.values()))

        random_view = scenario_views["Random-K"]
        step_map_randomk = allocate_exact_step_budget(
            selected=randomk_clients,
            client_arrays=random_view["client_arrays"],
            target_total=vr_target_per_round,
            batch_size=BATCH_SIZE,
            local_epochs=LOCAL_EPOCHS,
        )
        if int(sum(step_map_randomk.values())) != vr_target_per_round:
            raise RuntimeError(
                f"Exact Random-K/TADP optimizer-step parity failed at K={k}."
            )

        scenario_step_maps = {
            "Vanilla FedAvg": [
                dict(step_map_all) for _ in range(NUM_ROUNDS_FL)
            ],
            "Random-K": [
                dict(step_map_randomk) for _ in range(NUM_ROUNDS_FL)
            ],
            "TADP-VR Federated": [
                dict(step_map_vr) for _ in range(NUM_ROUNDS_FL)
            ],
        }

        frozen_plan = {
            "k_submissions": int(k),
            "partition_seed": int(partition_seed),
            "evidence_seed": int(evidence_seed),
            "randomk_seed": int(randomk_seed),
            "all_clients": all_clients,
            "tadp_vr_clients": vr_clients,
            "randomk_clients": randomk_clients,
            "observed_tadp_admitted": int(len(vr_clients)),
            "observed_tadp_admission_rate": float(len(vr_clients) / k),
            "tadp_vr_steps_per_round": int(vr_target_per_round),
            "randomk_steps_per_round": int(sum(step_map_randomk.values())),
            "fedavg_steps_per_round": int(sum(step_map_all.values())),
            "partition_integrity": data["integrity"],
            "scenario_input_dims": {
                s: int(v["input_dim"])
                for s, v in scenario_views.items()
            },
            "scenario_preprocessor_sha256": {
                s: str(v["preprocessor_sha256"])
                for s, v in scenario_views.items()
            },
            "preprocessing_fit_after_selection": True,
        }
        atomic_write_json(
            frozen_plan, root / f"frozen_plan_K{k:03d}.json"
        )

        for r in range(1, NUM_ROUNDS_FL + 1):
            parity_rows.append({
                "k_submissions": int(k),
                "round": int(r),
                "tadp_vr_k": int(len(vr_clients)),
                "randomk_k": int(len(randomk_clients)),
                "same_selected_k": bool(
                    len(vr_clients) == len(randomk_clients)
                ),
                "tadp_vr_steps": int(vr_target_per_round),
                "randomk_steps": int(sum(step_map_randomk.values())),
                "exact_step_parity": bool(
                    vr_target_per_round
                    == int(sum(step_map_randomk.values()))
                ),
                "fedavg_steps": int(sum(step_map_all.values())),
                "tadp_vr_clients": ";".join(vr_clients),
                "randomk_clients": ";".join(randomk_clients),
            })

        k_design_rows.append({
            "k_submissions": int(k),
            "partition_seed": int(partition_seed),
            "evidence_seed": int(evidence_seed),
            "randomk_seed": int(randomk_seed),
            "submitted_clients": int(k),
            "tadp_vr_admitted": int(len(vr_clients)),
            "tadp_vr_admission_rate": float(len(vr_clients) / k),
            "randomk_selected": int(len(randomk_clients)),
            "fedavg_clients": int(len(all_clients)),
            "vr_steps_per_round": int(vr_target_per_round),
            "randomk_steps_per_round": int(sum(step_map_randomk.values())),
            "fedavg_steps_per_round": int(sum(step_map_all.values())),
            "fedavg_input_dim": int(
                scenario_views["Vanilla FedAvg"]["input_dim"]
            ),
            "randomk_input_dim": int(
                scenario_views["Random-K"]["input_dim"]
            ),
            "tadp_vr_input_dim": int(
                scenario_views["TADP-VR Federated"]["input_dim"]
            ),
        })

        for run_idx, seed in enumerate(TRAINING_RUN_SEEDS, start=1):
            for scenario_idx, scenario in enumerate(SCENARIOS, start=1):
                key = f"K{k}|seed{seed}|{scenario}"
                if key in completed:
                    print(f"CHECKPOINT FOUND -> SKIP: {key}")
                    continue

                view = scenario_views[scenario]
                selected = scenario_clients[scenario]

                # Same training seed is used across scenarios. If two scenarios
                # have the same input dimension, this yields identical W0.
                seed_everything(seed)
                build_model_fn = (
                    lambda dim=int(view["input_dim"]):
                    build_diabetes_model(dim, lr=LEARNING_RATE)
                )
                base_model = build_model_fn()
                initial_weights = [
                    np.array(w, copy=True)
                    for w in base_model.get_weights()
                ]
                initial_hash = sha256_weights(initial_weights)
                del base_model
                tf.keras.backend.clear_session()

                train_monitor = build_train_monitor_subset(
                    view["client_arrays"],
                    max_samples=4096,
                    seed=99117 + k,
                )

                result = federated_train(
                    build_model_fn=build_model_fn,
                    initial_weights=initial_weights,
                    selected_per_round=[
                        list(selected)
                        for _ in range(NUM_ROUNDS_FL)
                    ],
                    client_arrays=view["client_arrays"],
                    X_test=view["X_test"],
                    y_test=view["y_test"],
                    n_classes=3,
                    batch_size=BATCH_SIZE,
                    local_epochs=LOCAL_EPOCHS,
                    class_weights=view["class_weights"],
                    run_seed=seed,
                    exact_step_maps=scenario_step_maps[scenario],
                    equal_weight=False,
                    fedprox_mu=0.0,
                    train_monitor=train_monitor,
                    progress_context={
                        "scenario": f"K={k} | {scenario}",
                        "run_idx": run_idx,
                        "run_total": len(TRAINING_RUN_SEEDS),
                        "scenario_idx": scenario_idx,
                        "scenario_total": len(SCENARIOS),
                        "overall_idx": (
                            (
                                PREDICTIVE_CLIENT_COUNTS.index(k)
                                * len(TRAINING_RUN_SEEDS)
                                + (run_idx - 1)
                            )
                            * len(SCENARIOS)
                            + scenario_idx
                        ),
                        "overall_total": (
                            len(PREDICTIVE_CLIENT_COUNTS)
                            * len(TRAINING_RUN_SEEDS)
                            * len(SCENARIOS)
                        ),
                    },
                )

                row = result_row(
                    run_idx, seed, scenario, result, initial_hash
                )
                row.update({
                    "k_submissions": int(k),
                    "partition_seed": int(partition_seed),
                    "evidence_seed": int(evidence_seed),
                    "randomk_seed": int(randomk_seed),
                    "selected_client_ids": ";".join(selected),
                    "selected_k": int(len(selected)),
                    "fl_rounds": int(NUM_ROUNDS_FL),
                    "local_epochs": int(LOCAL_EPOCHS),
                    "batch_size": int(BATCH_SIZE),
                    "learning_rate": float(LEARNING_RATE),
                    "dirichlet_alpha": float(DIRICHLET_ALPHA),
                    "vr_target_steps_per_round": int(
                        vr_target_per_round
                    ),
                    "scenario_steps_per_round": int(
                        sum(scenario_step_maps[scenario][0].values())
                    ),
                    "input_dim": int(view["input_dim"]),
                    "preprocessor_sha256": str(
                        view["preprocessor_sha256"]
                    ),
                    "training_rows": int(view["training_rows"]),
                    "class_weights_json": json.dumps(
                        {
                            str(a): float(b)
                            for a, b in view["class_weights"].items()
                        },
                        sort_keys=True,
                    ),
                    "preprocessing_fit_after_selection": True,
                    "comparison_role": (
                        "FULL_PARTICIPATION_REFERENCE"
                        if scenario == "Vanilla FedAvg"
                        else "MATCHED_RANDOM_CONTROL"
                        if scenario == "Random-K"
                        else "TADP_GOVERNANCE_SELECTION"
                    ),
                })

                print(
                    f"FINAL TEST | K={k} | {scenario} | "
                    f"Acc={row['accuracy']:.6f} | "
                    f"F1={row['f1_macro']:.6f} | "
                    f"AUC={row['roc_auc_ovr_macro']:.6f} | "
                    f"runtime={row['runtime_s']:.2f}s | "
                    f"communication={row['communication_mb']:.3f} MB | "
                    f"input_dim={row['input_dim']}"
                )

                upsert_csv(
                    row,
                    perf_checkpoint,
                    ["k_submissions", "seed", "scenario"],
                )
                write_b3_scenario_checkpoint(
                    root, k, run_idx, scenario, result
                )
                mark_checkpoint_complete(
                    checkpoint_state,
                    key,
                    extra={
                        "experiment_version": EXPERIMENT_VERSION,
                        "last_k": int(k),
                        "last_seed": int(seed),
                        "last_scenario": scenario,
                    },
                )
                completed.add(key)
                del result
                tf.keras.backend.clear_session()
                gc.collect()

    governance_df = pd.DataFrame(
        governance_metric_rows
    ).sort_values("k_submissions")
    governance_df.to_csv(
        root / "governance_scalability_metrics_K10_K20_K50_K100.csv",
        index=False,
    )

    parity_df = pd.DataFrame(parity_rows)
    parity_df.to_csv(
        root / "vr_randomk_exact_parity_by_K_round.csv",
        index=False,
    )
    if not parity_df.empty:
        if not parity_df["same_selected_k"].all():
            raise RuntimeError(
                "TADP-VR/Random-K selected-K parity failed."
            )
        if not parity_df["exact_step_parity"].all():
            raise RuntimeError(
                "TADP-VR/Random-K optimizer-step parity failed."
            )

    pd.DataFrame(k_design_rows).to_csv(
        root / "predictive_K_design_summary.csv", index=False
    )

    if all_preprocessing_audits:
        pd.concat(
            all_preprocessing_audits, ignore_index=True
        ).to_csv(
            root / "all_scenario_preprocessing_audit.csv",
            index=False,
        )

    perf = pd.read_csv(perf_checkpoint)
    expected_rows = (
        len(PREDICTIVE_CLIENT_COUNTS)
        * len(TRAINING_RUN_SEEDS)
        * len(SCENARIOS)
    )
    if len(perf) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} completed predictive rows, "
            f"found {len(perf)}"
        )

    # Initialization audit: identical W0 is required only where input dimensions
    # match. Option-A scenario-specific vocabularies can legitimately change
    # input dimension across cohorts.
    init_rows = []
    for k in PREDICTIVE_CLIENT_COUNTS:
        for seed in TRAINING_RUN_SEEDS:
            d = perf[
                perf["k_submissions"].eq(k)
                & perf["seed"].eq(seed)
            ].copy()
            for input_dim, g in d.groupby("input_dim", sort=True):
                hashes = g["initial_weights_sha256"].astype(str).unique()
                passed = len(hashes) == 1
                init_rows.append({
                    "k_submissions": int(k),
                    "seed": int(seed),
                    "input_dim": int(input_dim),
                    "scenarios": ";".join(g["scenario"].astype(str)),
                    "unique_initial_weight_hashes": int(len(hashes)),
                    "same_W0_when_input_dim_matches": bool(passed),
                    "initial_weights_sha256": (
                        str(hashes[0]) if len(hashes) else ""
                    ),
                })
                if not passed:
                    raise RuntimeError(
                        f"W0 parity failed for K={k}, seed={seed}, "
                        f"input_dim={input_dim}"
                    )

            tadp = d[
                d["scenario"].eq("TADP-VR Federated")
            ].iloc[0]
            rand = d[d["scenario"].eq("Random-K")].iloc[0]
            if int(tadp["selected_k"]) != int(rand["selected_k"]):
                raise RuntimeError(
                    f"Selected-K parity failed at K={k}, seed={seed}"
                )
            if int(tadp["optimizer_steps"]) != int(rand["optimizer_steps"]):
                raise RuntimeError(
                    f"Optimizer-step parity failed at K={k}, seed={seed}"
                )

    pd.DataFrame(init_rows).to_csv(
        root / "within_seed_initialization_and_parity_audit.csv",
        index=False,
    )

    summary = summarize_b3(perf)
    summary.to_csv(
        root / "predictive_scalability_summary_mean_sd_ci95.csv",
        index=False,
    )
    paired = paired_b3(perf)
    paired.to_csv(
        root / "paired_utility_differences.csv",
        index=False,
    )
    retention = utility_retention_table(summary)
    retention.to_csv(
        root / "utility_retention_vs_fedavg.csv",
        index=False,
    )

    print_banner("FINAL GOVERNANCE SCALABILITY METRICS")
    print(governance_df.to_string(index=False))

    print_banner("FINAL PREDICTIVE SCALABILITY SUMMARY")
    show = [
        "k_submissions", "scenario", "n_runs",
        "accuracy_mean", "accuracy_sd",
        "f1_macro_mean", "f1_macro_sd",
        "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_sd",
        "runtime_s_mean", "communication_mb_mean",
        "optimizer_steps_mean", "participants_mean",
    ]
    print(
        summary[
            [c for c in show if c in summary.columns]
        ].to_string(index=False)
    )

    print_banner("FINAL FAIL-CLOSED AUDITS")
    print("Global patient leakage audit: PASS")
    print("Partition row-integrity audit: PASS")
    print("Final 28-factor evidence-validity policy: PASS")
    print("Legacy mandatory-factor / Review Score policy removed: PASS")
    print("Governance/DQ occurs before model preprocessing: PASS")
    print("Scenario preprocessing fitted only after cohort selection: PASS")
    print("Scenario class weights fitted only after cohort selection: PASS")
    print("Documentary profile composition held comparable across K: PASS")
    print("Final TADP admission count is observed, not forced: PASS")
    print("Random-K uses observed TADP admitted count: PASS")
    print("TADP-VR / Random-K exact optimizer-step parity: PASS")
    print("Four federated rounds used in predictive scalability: PASS")
    print("No universal 1.12 communication multiplier is used: PASS")
    print("Held-out TEST used only for final predictive evaluation: PASS")
    print(f"Completed predictive runs: {expected_rows}")

    note = (
        "Experiment B predictive scalability evaluates K=20,50,100 using "
        "three training seeds and four FL rounds. K=10 is not retrained here; "
        "it is included only in the governance/ledger scalability measurement. "
        "The final Option-A design fits model preprocessing and class weights "
        "after participation is determined for each scenario. Therefore input "
        "dimension can legitimately differ between full FedAvg, Random-K, and "
        "TADP-VR when their selected cohorts expose different categorical "
        "vocabularies. For a K=10/20/50/100 predictive figure, combine the "
        "finalized Experiment-A1 K=10 results only after confirming metric and "
        "resource-accounting compatibility; wall-clock runtime from different "
        "hardware sessions should not be treated as directly comparable."
    )
    (root / "manuscript_integration_note.txt").write_text(
        note, encoding="utf-8"
    )
    return root


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(finished_root, EXPERIMENT_VERSION)




TADP-B-FINAL-v18.0-A1A2-OPTIONA-SCALABILITY-K20-50-100-3SEED-4ROUND
Governance K values: [10, 20, 50, 100]
Predictive K values: [20, 50, 100]
Training seeds: [42, 142, 242]
FL rounds: 4
Final policy: evidence-validity gate -> dimension floor -> HPS bands -> critical-dimension HPS_review -> binary ACCEPT/REJECT
Option A: governance/selection precedes model preprocessing; each scenario fits preprocessing and class weights only from its participating TRAIN cohort.
K=10 is governance-only here; predictive K=10 remains Experiment A1.
Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_B_TADP-B-FINAL-v18.0-A1A2-OPTIONA-SCALABILITY-K20-50-100-3SEED-4ROUND

GOVERNANCE / SCALABILITY PREPARATION — K=10
K=10: observed TADP admission=6/10 (60.0%); governance=5.1821s; ledger=4120 bytes.

GOVERNANCE / SCALABILITY PREPARATION — K=20
K=20: observed TADP admission=12/20 (60.0%); governance=5.2752s; ledger=8105 bytes.